# 09 — Spark SQL e Joins

**EP · Data Science Knowledge Base | Big Data e Processamento Distribuído | Notebook 09/16**

**Execução:** use o kernel Python 3, reinicie o kernel e execute as células em ordem. Laboratórios Spark são opcionais e desativados por padrão.

**Abordagem:** os exemplos locais verificam conceitos e não representam um cluster distribuído.


## Objetivos de aprendizagem
Comparar agregações locais e distribuídas.

Ao final, você deve conseguir interpretar os resultados, justificar as escolhas e explicar suas limitações.

## Fundamentação teórica
JOIN, GROUP BY, registros nulos, duplicatas e cardinalidade.

**Atenção metodológica:** execução local facilita entendimento e depuração, mas não comprova ganhos de paralelismo em infraestrutura distribuída. Escalabilidade exige medir dados, nós, rede, memória, partições e operações de shuffle.

In [1]:
from pathlib import Path
import sys
DATASETS = Path('datasets')
assert DATASETS.is_dir(), 'Execute a partir da raiz da área 10 (pasta datasets não encontrada)'
print('Python:', sys.version.split()[0])
print('Pasta dos dados:', DATASETS.as_posix())

Python: 3.14.4
Pasta dos dados: datasets


## Demonstração 1 — construção e observação

In [2]:
import pandas as pd
v = pd.read_csv('datasets/flights.csv')
a = pd.read_csv('datasets/airports.csv')
assert {'dest','carrier','distance'}.issubset(v)
assert {'faa','name'}.issubset(a)
a = a.dropna(subset=['faa']).copy()
conflitos = a.groupby('faa',dropna=False)['name'].nunique(dropna=False)
chaves_ambiguas = set(conflitos[conflitos>1].index)
a = a[~a['faa'].isin(chaves_ambiguas)].drop_duplicates(subset=['faa'],keep='first')
print('Códigos FAA ambíguos excluídos do cadastro para este join:',len(chaves_ambiguas))
joined = v.merge(a[['faa','name']],left_on='dest',right_on='faa',how='left',validate='many_to_one')
assert len(joined)==len(v)
print('Destinos sem correspondência:',int(joined['name'].isna().sum()))
print(joined[['dest','name']].head())
print(joined.groupby('carrier')['distance'].mean().sort_values(ascending=False).head())


Códigos FAA ambíguos excluídos do cadastro para este join: 1
Destinos sem correspondência: 0
  dest                           name
0  LAX               Los Angeles Intl
1  HNL                  Honolulu Intl
2  SFO             San Francisco Intl
3  SJC  Norman Y Mineta San Jose Intl
4  BUR                       Bob Hope
carrier
HA    2630.876712
AA    1739.230290
B6    1635.308411
DL    1566.137708
US    1525.512262
Name: distance, dtype: float64


## Interpretação
Observe o esquema, as unidades e os resultados. Antes de concluir que um algoritmo é eficiente, distinga **complexidade computacional**, **custo de comunicação** e **volume de dados**. Os cálculos locais ilustram a semântica; o Spark, quando disponível, ilustra a API distribuída.

## Demonstração 2 — validação e limitações

In [3]:
assert len(joined)==len(v)
print('Cardinalidade preservada:',len(joined))
print('Destinos sem cadastro:',joined['name'].isna().sum())
print('Em Spark SQL, verifique broadcast joins e estratégias físicas com explain().')

Cardinalidade preservada: 10000
Destinos sem cadastro: 0
Em Spark SQL, verifique broadcast joins e estratégias físicas com explain().


## Boas práticas e limitações

- Execute com versões registradas, datasets identificados e caminhos relativos.
- Diferencie execução local, emulação lógica e infraestrutura distribuída real.
- Documente nulos, chaves, cardinalidade, custos e limitações das métricas.
- Não publique datasets ou resultados cuja redistribuição não esteja autorizada.
- Evite `collect()` de grandes tabelas e não confunda uma execução `local[*]` com cluster multinó.


## Exercícios
1. Explique que parte do exemplo poderia ser distribuída por chave ou partição e qual é o custo do shuffle.
2. Descreva uma validação de qualidade para os dados usados.
3. Indique um cenário em que o exemplo local deixaria de ser adequado.

## Respostas comentadas (orientativas)
1. Agregações por chave podem combinar resultados parciais, mas exigem redistribuir dados com a mesma chave; filtros e projeções usualmente evitam shuffle.
2. Verifique contagem, unicidade de chaves, percentual de valores ausentes, tipos, intervalos e consistência das unidades.
3. Quando dados, latência, concorrência ou exigências de falha excederem a capacidade de uma máquina; nesse caso, considere Spark/Hadoop e meça os custos adicionais.

## Síntese

O capítulo 09 reuniu conceitos e verificações de **cardinalidade e semântica de junções**. Os exemplos de pequena escala não comprovam desempenho em cluster. Consulte apenas referências complementares efetivamente publicadas e documentadas.


## Revisão científica — auditoria e aplicação

**Ponto de atenção deste capítulo:** joins e cardinalidade. Esta seção complementa os exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **joins e cardinalidade** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — SQL: agregação com SQLite local (comparável ao Spark SQL)

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.

In [4]:
import sqlite3
con=sqlite3.connect(':memory:')
try:
 con.execute('CREATE TABLE viagens (regiao TEXT, km INTEGER)')
 con.executemany('INSERT INTO viagens VALUES (?,?)',[('N',10),('N',20),('S',15)])
 print(con.execute('SELECT regiao, SUM(km) FROM viagens GROUP BY regiao ORDER BY regiao').fetchall())
finally:
 con.close()

[('N', 30), ('S', 15)]


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **joins e cardinalidade** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Laboratório opcional — Spark local real

Requer `RUN_SPARK_LABS=1` e `SPARK_LAB_PYTHON` apontando para um ambiente isolado com PySpark e Java. O teste falha explicitamente se executado incorretamente; resultados locais não comprovam cluster distribuído.


In [5]:
# Laboratório Spark real opcional, executado em interpretador isolado.
# Ative SOMENTE quando o ambiente PySpark e o Java já estiverem preparados.
import os, sys, subprocess, textwrap
if os.environ.get('RUN_SPARK_LABS') == '1':
    interpretador = os.environ.get('SPARK_LAB_PYTHON', sys.executable)
    assert os.path.isfile(interpretador), f'Interpretador não encontrado: {interpretador}'
    programa = textwrap.dedent(r'''from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,StringType,DoubleType
import pandas as pd
s=SparkSession.builder.master('local[2]').appName('ep-area10-09').config('spark.ui.enabled','false').getOrCreate()
try:
    s.sparkContext.setLogLevel('ERROR')
    voos=pd.read_csv('datasets/flights.csv').head(300)
    aeroportos=pd.read_csv('datasets/airports.csv').dropna(subset=['faa'])
    conflitos=aeroportos.groupby('faa')['name'].nunique(dropna=False)
    chaves_ambiguas=set(conflitos[conflitos>1].index)
    aeroportos=aeroportos[~aeroportos.faa.isin(chaves_ambiguas)].drop_duplicates('faa')
    assert not aeroportos['faa'].duplicated().any()
    vv=[(str(x.carrier),str(x.dest),float(x.distance)) for x in voos.itertuples()]
    aa=[(str(x.faa),str(x.name)) for x in aeroportos.itertuples()]
    vs=s.createDataFrame(vv,StructType([StructField('carrier',StringType()),StructField('dest',StringType()),StructField('distance',DoubleType())]))
    ars=s.createDataFrame(aa,['faa','name'])
    vs.createOrReplaceTempView('voos');ars.createOrReplaceTempView('aeroportos')
    n=s.sql('SELECT v.dest, a.name FROM voos v LEFT JOIN aeroportos a ON v.dest=a.faa').count()
    assert n==len(voos),(n,len(voos))
    print('Spark SQL LEFT JOIN, cardinalidade preservada:',n)
    print('SPARK_LAB_OK_09')
finally:s.stop()''')
    ambiente = os.environ.copy()
    ambiente.setdefault('SPARK_LOCAL_IP', '127.0.0.1')
    ambiente['PYSPARK_PYTHON'] = interpretador
    ambiente['PYSPARK_DRIVER_PYTHON'] = interpretador
    resultado = subprocess.run([interpretador, '-c', programa], cwd=os.getcwd(),
                               env=ambiente, text=True, capture_output=True, timeout=150)
    if resultado.stdout: print(resultado.stdout.strip())
    if resultado.returncode:
        print(resultado.stderr[-3500:])
        raise RuntimeError('O LABORATÓRIO SPARK 09 FALHOU; não registrar como aprovado.')
    assert 'SPARK_LAB_OK_09' in resultado.stdout, 'Marcador de sucesso Spark não encontrado'
else:
    print('SPARK NÃO EXECUTADO (opcional). Ative RUN_SPARK_LABS=1 e SPARK_LAB_PYTHON.')


Spark SQL LEFT JOIN, cardinalidade preservada: 300
SPARK_LAB_OK_09
